## **Read from csv and create Data Frame**

In [13]:
import pandas as pd
df=pd.read_csv("synthetic_logs.csv")

In [14]:
df.head()

,timestamp,source,log_message,target_label
0,27-06-2025 07:20,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert
3,12-07-2025 00:24,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status
4,02-06-2025 18:25,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status


In [15]:
df.target_label.unique()

<StringArray>
[        'HTTP Status',      'Critical Error',      'Security Alert',
               'Error', 'System Notification',      'Resource Usage',
         'User Action',      'Workflow Error', 'Deprecation Warning']
Length: 9, dtype: str

## **Create vecors for each tokens**

In [16]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# Your log data
logs = df["log_message"].tolist()

# Convert text → vectors
vectorizer = TfidfVectorizer(
    max_features=2000, 
    ngram_range=(1,2),
    stop_words=None
)

log_vectors = vectorizer.fit_transform(logs)


In [17]:
type(log_vectors)

scipy.sparse._csr.csr_matrix

In [18]:
log_vectors[1].toarray()

array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 2000))

In [19]:
uniq_types = df.target_label.unique()

In [20]:
print(uniq_types.size)

9


## **Clustering the messages**

In [21]:
# Clustering
# As uniq_types is 9 we are chosing 10 clusters

kmeans = KMeans(n_clusters=10, random_state=42)
df["cluster"] = kmeans.fit_predict(log_vectors)



In [22]:
df.head()

,timestamp,source,log_message,target_label,cluster
0,27-06-2025 07:20,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,0
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,7
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,7
3,12-07-2025 00:24,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,0
4,02-06-2025 18:25,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,0


In [23]:
df.cluster.unique()

array([0, 7, 3, 6, 2, 5, 8, 9, 1, 4], dtype=int32)

In [24]:
df[df.cluster == 8]

,timestamp,source,log_message,target_label,cluster
18,2/22/2025 17:49,ModernCRM,Account with ID 5351 created by User634.,User Action,8
27,9/24/2025 19:57,ThirdPartyAPI,User User685 logged out.,User Action,8
36,11/19/2025 13:14,BillingSystem,System reboot initiated by user User243.,System Notification,8
57,9/14/2025 3:03,AnalyticsEngine,User User395 logged in.,User Action,8
85,3/13/2025 2:11,ModernHR,User User225 logged in.,User Action,8
...,...,...,...,...,...
2275,3/13/2025 17:17,AnalyticsEngine,User User755 logged out.,User Action,8
2317,03-07-2025 05:44,BillingSystem,System reboot initiated by user User724.,System Notification,8
2323,12-01-2025 18:17,ThirdPartyAPI,User User882 logged out.,User Action,8
2353,06-10-2025 19:38,AnalyticsEngine,User User536 logged in.,User Action,8


## **Calsify the logs using RegX**

In [28]:
import re

# Define patterns (pattern, label)
patterns = [
    (r"\b(GET|POST|PUT|DELETE|PATCH)\b.*\b(200|201|204|301|302|400|401|403|404|500|502|503)\b", "HTTP_STATUS"),

    (r"\b(login|logged in|signin|authenticated).*(success|successful)\b", "USER_ACTION"),

    (r"\b(login|authentication|signin).*(fail|failed|error|unsuccessful|denied)\b", "SECURITY_ALERT"),

    (r"\b(access|permission|authorization).*(denied|forbidden|unauthorized|rejected)\b", "SECURITY_ALERT"),

    (r"\b(db|database).*(fail|error|timeout|unreachable|lost|issue)\b", "ERROR"),

    (r"\b(exception|fatal).*(null|undefined|pointer|crash)?\b", "CRITICAL_ERROR"),

    (r"\b(memory|cpu|disk|resource).*(high|low|exceed|spike|leak|full|usage)\b", "RESOURCE_USAGE"),

    (r"\b(service|server|application).*(start|started|stop|stopped|restart|restarted)\b", "SYSTEM_NOTIFICATION"),

    (r"\b(file|resource|document).*(not found|missing|unavailable)\b", "ERROR"),

    (r"\b(timeout|timed out|took too long|delayed)\b", "ERROR"),

    (r"\b(api|endpoint|service).*(fail|error|unavailable|down)\b", "ERROR"),

    (r"\b(payment|transaction|billing|invoice).*(fail|error|declined|pending)\b", "WORKFLOW_ERROR"),

    (r"\b(deprecated|obsolete|no longer supported|will be removed)\b", "DEPRECATION_WARNING"),

    (r"\b(config|configuration|setting).*(invalid|missing|error|incorrect)\b", "ERROR"),

    (r"\b(suspicious|attack|intrusion|malicious|threat)\b", "SECURITY_ALERT"),

    (r"\b(job|task|queue|process).*(start|started|complete|completed|fail|failed)\b", "SYSTEM_NOTIFICATION"),

    (r"\b(cache).*(miss|fail|error|evict|clear)\b", "RESOURCE_USAGE"),
]

# Compile all patterns once
compiled_patterns = [(re.compile(p, re.IGNORECASE), label) for p, label in patterns]


def classify(log_message: str) -> str:
    for pattern, label in compiled_patterns:
        if pattern.search(log_message):
            return label
    return "UNKNOWN"

In [ ]:
if __name__ == "__main__":
    logs = [
        "GET /api/users HTTP/1.1 200",
        "User john logged in successfully",
        "Login failed due to invalid credentials",
        "Database connection timeout error",
        "Out of memory error in service",
        "Payment failed for transaction 12345",
        "Suspicious activity detected from IP 192.168.1.1"
    ]

    for log in logs:
        print(f"{log}  --->  {classify(log)}")

GET /api/users HTTP/1.1 200  --->  HTTP_STATUS
User john logged in successfully  --->  UNKNOWN
Login failed due to invalid credentials  --->  SECURITY_ALERT
Database connection timeout error  --->  ERROR
Out of memory error in service  --->  UNKNOWN
Payment failed for transaction 12345  --->  UNKNOWN
Suspicious activity detected from IP 192.168.1.1  --->  SECURITY_ALERT


In [30]:
logs = [
    "System initialized successfully",
    "Login attempt unsuccessful",
    "Latency recorded at 200ms",
    "Order created successfully"
]

for log in logs:
    print(classify(log))

UNKNOWN
USER_ACTION
UNKNOWN
UNKNOWN


In [31]:
df['regx_labels'] = df['log_message'].apply(classify)

In [32]:
df_non_regx = df[df['regx_labels']=="UNKNOWN"].copy()

In [33]:
df_non_regx.shape

(1300, 6)

## **Classify the logs that has less training data**

In [34]:
# Get target_labels that have fewer than 5 rows
label_counts = df['target_label'].value_counts()
rare_labels = label_counts[label_counts < 5].index


In [35]:

rare_labels
type(df)

pandas.DataFrame

In [36]:
training_df = df[~df['target_label'].isin(rare_labels)]


In [37]:
training_df.shape

(2403, 6)

In [ ]:
from fastembed import TextEmbedding
import numpy as np

model = TextEmbedding(model_name="BAAI/bge-small-en")

log_messages = training_df["log_message"].tolist()

embeddings = np.array(list(model.embed(log_messages)))


In [39]:
embeddings[0]

array([-8.37230757e-02, -5.99146442e-05, -6.89718733e-03, -4.57780883e-02,
        1.24226091e-02, -1.70555431e-02, -1.69710349e-02, -2.31087990e-02,
        1.26650427e-02, -7.44740758e-03,  9.16670263e-03, -4.88223732e-02,
        4.20215391e-02,  1.76765379e-02,  1.37700886e-02,  1.87685788e-02,
       -1.06409835e-02, -1.49161378e-02, -4.63518896e-04,  1.00147948e-02,
        4.85533997e-02, -4.79787476e-02, -1.28038684e-02, -1.87198818e-02,
       -7.66243488e-02,  4.90789860e-02, -1.21257622e-02, -1.96023881e-02,
       -2.53855325e-02, -1.84138536e-01, -3.95326428e-02, -5.17901406e-02,
       -2.85508614e-02, -2.39621177e-02,  2.92274319e-02, -2.80292388e-02,
       -1.16248708e-02,  1.82595141e-02, -2.30680145e-02,  4.86318544e-02,
        6.81315958e-02,  1.15745338e-02, -3.06101441e-02, -9.72947478e-03,
       -1.65466238e-02, -4.20337729e-02, -2.30020080e-02, -2.70570000e-03,
        1.50465397e-02, -5.78415440e-03, -8.70369934e-03,  1.09547088e-02,
       -2.94803968e-03,  

In [40]:
embeddings.shape

(2403, 384)

In [52]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = embeddings
y = training_df["target_label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2, stratify=y
)

clf_model = LogisticRegression(max_iter=1000, random_state=2)
clf_model.fit(X_train, y_train)

y_pred = clf_model.predict(X_test)



In [53]:

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy ", accuracy)


Accuracy  0.9916839916839917


In [58]:
import joblib

joblib.dump(clf_model, "../log_classifier.joblib")


['../log_classifier.joblib']